In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd
import random
import nltk
from nltk.corpus import wordnet
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:

df = pd.read_csv(
    'WELFake_original_small.csv',
    encoding='utf-8',
    on_bad_lines='skip',
    engine='python'
)

df = df[['text', 'label']].dropna()

print(df.shape)
print(df['label'].value_counts())

(6000, 2)
label
0    3000
1    3000
Name: count, dtype: int64


In [ ]:
import re

def clean_text(text):
    # odstráni HTML tagy
    text = re.sub(r'<.*?>', ' ', text)

    # odstráni len špeciálne znaky (NECHÁ písmená, čísla a interpunkciu)
    text = re.sub(r'[^a-zA-Z0-9\s\.\!\?]', ' ', text)

    # viacnásobné medzery → jedna
    text = re.sub(r'\s+', ' ', text)

    return text.strip().lower()

In [ ]:
import pandas as pd

# odstránenie NaN
df = df.dropna(subset=['text'])

# odstránenie prázdnych stringov
df = df[df['text'].str.strip() != '']

# zabezpečenie string typu
df['text'] = df['text'].astype(str)

# aplikovanie clean_text
df['text'] = df['text'].apply(clean_text)

In [ ]:
samples_per_class = min(500, df['label'].value_counts().min())

df_small = df.groupby('label', group_keys=False).apply(
    lambda x: x.sample(samples_per_class)
)

print(df_small['label'].value_counts())
df.head()

/tmp/ipykernel_894/784443835.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_small = df.groupby('label', group_keys=False).apply(


label
0    500
1    500
Name: count, dtype: int64


,text,label
0,washington reuters u.s. president donald trump...,0
1,beirut reuters the syrian government envoy to ...,0
2,tokyo reuters born and raised in japan three n...,0
3,tokyo reuters nato chief jens stoltenberg urge...,0
4,chicago washington reuters the u.s. department...,0


In [ ]:
df_small = pd.read_csv(
    'WELFake_original_small_2.csv',
    encoding='utf-8',
    on_bad_lines='skip',
    engine='python'
)

In [ ]:
!pip install transformers sentencepiece torch

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

In [ ]:
import torch
from tqdm import tqdm

In [ ]:


device = "cuda" if torch.cuda.is_available() else "cpu"

# EN → DE
model_name_en_de = "Helsinki-NLP/opus-mt-en-de"
tokenizer_en_de = MarianTokenizer.from_pretrained(model_name_en_de)
model_en_de = MarianMTModel.from_pretrained(model_name_en_de).to(device)

# DE → EN
model_name_de_en = "Helsinki-NLP/opus-mt-de-en"
tokenizer_de_en = MarianTokenizer.from_pretrained(model_name_de_en)
model_de_en = MarianMTModel.from_pretrained(model_name_de_en).to(device)


def translate_batch(texts, model, tokenizer, batch_size=16, desc="Translating"):
    results = []

    for i in tqdm(range(0, len(texts), batch_size), desc=desc):
        batch = texts[i:i+batch_size]

        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(device)

        with torch.no_grad():
            translated = model.generate(**inputs)

        decoded = tokenizer.batch_decode(translated, skip_special_tokens=True)
        results.extend(decoded)

    return results


def back_translation_batch(texts, batch_size=16):
    german = translate_batch(texts, model_en_de, tokenizer_en_de, batch_size, desc="EN → DE")
    back_translated = translate_batch(german, model_de_en, tokenizer_de_en, batch_size, desc="DE → EN")
    return back_translated

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/768k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/797k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/298M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/298M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np
import random

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

nb_scores = {"acc": [], "prec": [], "rec": [], "f1": []}
lr_scores = {"acc": [], "prec": [], "rec": [], "f1": []}

for fold, (train_idx, test_idx) in enumerate(kf.split(df_small, df_small['label'])):
    print(f"\nFold {fold+1}")
    random.seed(42 + fold)

    train_df = df_small.iloc[train_idx].copy()
    test_df = df_small.iloc[test_idx].copy()


    #  AUGMENTÁCIA LEN TRAIN
    sample_size = int(0.2 * len(train_df))

    aug_df = train_df.sample(sample_size, replace=True, random_state=42).copy()

    # batch back-translation
    texts = aug_df['text'].tolist()
    aug_texts = back_translation_batch(texts, batch_size=16)

    aug_df['text'] = aug_texts

    train_aug = pd.concat([train_df, aug_df])

    # TF-IDF
    vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')

    X_train = vectorizer.fit_transform(train_aug['text'])
    y_train = train_aug['label'].astype(int)

    X_test = vectorizer.transform(test_df['text'])
    y_test = test_df['label'].astype(int)

    # MODELY
    nb = MultinomialNB()
    lr = LogisticRegression(max_iter=1000)

    nb.fit(X_train, y_train)
    lr.fit(X_train, y_train)

    y_pred_nb = nb.predict(X_test)
    y_pred_lr = lr.predict(X_test)

    #  METRIKY
    def compute_metrics(y_true, y_pred):
        return {
            "acc": accuracy_score(y_true, y_pred),
            "prec": precision_score(y_true, y_pred, average='weighted', zero_division=0),
            "rec": recall_score(y_true, y_pred, average='weighted', zero_division=0),
            "f1": f1_score(y_true, y_pred, average='weighted', zero_division=0),
        }

    nb_m = compute_metrics(y_test, y_pred_nb)
    lr_m = compute_metrics(y_test, y_pred_lr)

    for k in nb_scores:
        nb_scores[k].append(nb_m[k])
        lr_scores[k].append(lr_m[k])

#  VÝSLEDKY
print("\nNaive Bayes:")
for k in nb_scores:
    print(f"{k.upper()}: {nb_scores[k]}")
    print(f"Average {k.upper()}: {np.mean(nb_scores[k])}")

print("\nLogistic Regression:")
for k in lr_scores:
    print(f"{k.upper()}: {lr_scores[k]}")
    print(f"Average {k.upper()}: {np.mean(lr_scores[k])}")


Fold 1


DE → EN: 100%|██████████| 10/10 [02:55<00:00, 17.52s/it]



Fold 2


DE → EN: 100%|██████████| 10/10 [02:53<00:00, 17.38s/it]



Fold 3


...


Fold 4


DE → EN: 100%|██████████| 10/10 [03:05<00:00, 18.54s/it]



Fold 5


DE → EN: 100%|██████████| 10/10 [02:53<00:00, 17.32s/it]



Naive Bayes:
ACC: [0.805, 0.835, 0.805, 0.83, 0.815]
Average ACC: 0.818
PREC: [0.8065018591096372, 0.8358395989974937, 0.8057644110275689, 0.8311922922521076, 0.8157894736842105]
Average PREC: 0.8190175270142035
REC: [0.805, 0.835, 0.805, 0.83, 0.815]
Average REC: 0.818
F1: [0.8047608320192237, 0.8348968105065666, 0.804878048780488, 0.8298468621759583, 0.8148843026891808]
Average F1: 0.8178533712342834

Logistic Regression:
ACC: [0.855, 0.855, 0.82, 0.85, 0.86]
Average ACC: 0.8480000000000001
PREC: [0.855035503550355, 0.855035503550355, 0.8201280512204883, 0.8522544283413848, 0.8613006824568447]
Average PREC: 0.8487508338238856
REC: [0.855, 0.855, 0.82, 0.85, 0.86]
Average REC: 0.8480000000000001
F1: [0.8549963749093727, 0.8549963749093727, 0.8199819981998201, 0.8497596153846153, 0.859873886497848]
Average F1: 0.8479216499802058


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

nb_scores = {"acc": [], "prec": [], "rec": [], "f1": []}
lr_scores = {"acc": [], "prec": [], "rec": [], "f1": []}

for fold, (train_idx, test_idx) in enumerate(kf.split(df_small, df_small['label'])):
    print(f"\nFold {fold+1}")
    random.seed(42 + fold)

    train_df = df_small.iloc[train_idx].copy()
    test_df = df_small.iloc[test_idx].copy()


    # AUGMENTÁCIA LEN TRAIN
    sample_size = int(0.5 * len(train_df))

    aug_df = train_df.sample(sample_size, replace=True, random_state=42).copy()

    # batch back-translation
    texts = aug_df['text'].tolist()
    aug_texts = back_translation_batch(texts, batch_size=16)

    aug_df['text'] = aug_texts

    train_aug = pd.concat([train_df, aug_df])

    # TF-IDF
    vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')

    X_train = vectorizer.fit_transform(train_aug['text'])
    y_train = train_aug['label'].astype(int)

    X_test = vectorizer.transform(test_df['text'])
    y_test = test_df['label'].astype(int)

    # MODELY
    nb = MultinomialNB()
    lr = LogisticRegression(max_iter=1000)

    nb.fit(X_train, y_train)
    lr.fit(X_train, y_train)

    y_pred_nb = nb.predict(X_test)
    y_pred_lr = lr.predict(X_test)

    # METRIKY
    def compute_metrics(y_true, y_pred):
        return {
            "acc": accuracy_score(y_true, y_pred),
            "prec": precision_score(y_true, y_pred, average='weighted', zero_division=0),
            "rec": recall_score(y_true, y_pred, average='weighted', zero_division=0),
            "f1": f1_score(y_true, y_pred, average='weighted', zero_division=0),
        }

    nb_m = compute_metrics(y_test, y_pred_nb)
    lr_m = compute_metrics(y_test, y_pred_lr)

    for k in nb_scores:
        nb_scores[k].append(nb_m[k])
        lr_scores[k].append(lr_m[k])

# VÝSLEDKY
print("\nNaive Bayes:")
for k in nb_scores:
    print(f"{k.upper()}: {nb_scores[k]}")
    print(f"Average {k.upper()}: {np.mean(nb_scores[k])}")

print("\nLogistic Regression:")
for k in lr_scores:
    print(f"{k.upper()}: {lr_scores[k]}")
    print(f"Average {k.upper()}: {np.mean(lr_scores[k])}")


Fold 1


DE → EN: 100%|██████████| 25/25 [07:33<00:00, 18.13s/it]



Fold 2


DE → EN: 100%|██████████| 25/25 [07:20<00:00, 17.63s/it]



Fold 3


...


Fold 4


DE → EN: 100%|██████████| 25/25 [07:42<00:00, 18.49s/it]



Fold 5


DE → EN: 100%|██████████| 25/25 [07:14<00:00, 17.38s/it]



Naive Bayes:
ACC: [0.79, 0.825, 0.815, 0.83, 0.815]
Average ACC: 0.8149999999999998
PREC: [0.7901160464185674, 0.8258145363408521, 0.8152837553798418, 0.8311922922521076, 0.8165511003919204]
Average PREC: 0.8157915461566578
REC: [0.79, 0.825, 0.815, 0.83, 0.815]
Average REC: 0.8149999999999998
F1: [0.7899789978997899, 0.8248905565978737, 0.8149583656322673, 0.8298468621759583, 0.8147730970438788]
Average F1: 0.8148895758699535

Logistic Regression:
ACC: [0.855, 0.84, 0.81, 0.87, 0.86]
Average ACC: 0.8470000000000001
PREC: [0.855035503550355, 0.8405448717948718, 0.81, 0.8713368125250903, 0.8623188405797102]
Average PREC: 0.8478472056900055
REC: [0.855, 0.84, 0.81, 0.87, 0.86]
Average REC: 0.8470000000000001
F1: [0.8549963749093727, 0.839935974389756, 0.81, 0.8698828946051447, 0.859775641025641]
Average F1: 0.846918176985983


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

nb_scores = {"acc": [], "prec": [], "rec": [], "f1": []}
lr_scores = {"acc": [], "prec": [], "rec": [], "f1": []}

for fold, (train_idx, test_idx) in enumerate(kf.split(df_small, df_small['label'])):
    print(f"\nFold {fold+1}")
    random.seed(42 + fold)

    train_df = df_small.iloc[train_idx].copy()
    test_df = df_small.iloc[test_idx].copy()


    # AUGMENTÁCIA LEN TRAIN
    sample_size = int(1 * len(train_df))

    aug_df = train_df.sample(sample_size, replace=True, random_state=42).copy()

    #batch back-translation
    texts = aug_df['text'].tolist()
    aug_texts = back_translation_batch(texts, batch_size=16)

    aug_df['text'] = aug_texts

    train_aug = pd.concat([train_df, aug_df])

    # TF-IDF
    vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')

    X_train = vectorizer.fit_transform(train_aug['text'])
    y_train = train_aug['label'].astype(int)

    X_test = vectorizer.transform(test_df['text'])
    y_test = test_df['label'].astype(int)

    # MODELY
    nb = MultinomialNB()
    lr = LogisticRegression(max_iter=1000)

    nb.fit(X_train, y_train)
    lr.fit(X_train, y_train)

    y_pred_nb = nb.predict(X_test)
    y_pred_lr = lr.predict(X_test)

    # METRIKY
    def compute_metrics(y_true, y_pred):
        return {
            "acc": accuracy_score(y_true, y_pred),
            "prec": precision_score(y_true, y_pred, average='weighted', zero_division=0),
            "rec": recall_score(y_true, y_pred, average='weighted', zero_division=0),
            "f1": f1_score(y_true, y_pred, average='weighted', zero_division=0),
        }

    nb_m = compute_metrics(y_test, y_pred_nb)
    lr_m = compute_metrics(y_test, y_pred_lr)

    for k in nb_scores:
        nb_scores[k].append(nb_m[k])
        lr_scores[k].append(lr_m[k])

# VÝSLEDKY
print("\nNaive Bayes:")
for k in nb_scores:
    print(f"{k.upper()}: {nb_scores[k]}")
    print(f"Average {k.upper()}: {np.mean(nb_scores[k])}")

print("\nLogistic Regression:")
for k in lr_scores:
    print(f"{k.upper()}: {lr_scores[k]}")
    print(f"Average {k.upper()}: {np.mean(lr_scores[k])}")


Fold 1


EN → DE:   0%|          | 0/50 [02:59<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

False
NO GPU


In [ ]:
import matplotlib.pyplot as plt
import numpy as np